In [7]:
import pandas as pd

df = pd.read_csv('titanic.csv')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
total_pasajeros = len(df)
ids_distintos = df['PassengerId'].nunique()

tabla_1 = pd.DataFrame({
    'Métrica': ['Total de registros en el archivo', 'PassengerId distintos', '¿Coinciden ambos valores?'],
    'Resultado': [total_pasajeros, ids_distintos, 'Sí' if total_pasajeros == ids_distintos else 'No']
})

display(tabla_1)
print("Tratamiento de faltantes: La columna PassengerId no presenta valores nulos (0 faltantes).")

In [ ]:
conteo_clase = df['Pclass'].value_counts().sort_index().reset_index()
conteo_clase.columns = ['Clase del Boleto', 'Cantidad de Pasajeros']
conteo_clase['Clase del Boleto'] = conteo_clase['Clase del Boleto'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'})

fila_total = pd.DataFrame({'Clase del Boleto': ['Total General'], 'Cantidad de Pasajeros': [conteo_clase['Cantidad de Pasajeros'].sum()]})
tabla_2 = pd.concat([conteo_clase, fila_total], ignore_index=True)

display(tabla_2)
print("Clase con más pasajeros registrados: 3ª Clase.")
print("Tratamiento de faltantes: La columna Pclass no contiene valores faltantes.")

In [ ]:

puerto_serie = df['Embarked'].fillna('Sin registrar').value_counts().reset_index()
puerto_serie.columns = ['Código Puerto', 'Cantidad de Pasajeros']

nombres_puertos = {'S': 'Southampton (S)', 'C': 'Cherburgo (C)', 'Q': 'Queenstown (Q)', 'Sin registrar': 'Sin registrar'}
puerto_serie['Puerto de Embarque'] = puerto_serie['Código Puerto'].map(nombres_puertos)
tabla_3 = puerto_serie[['Puerto de Embarque', 'Cantidad de Pasajeros']]

fila_total = pd.DataFrame({'Puerto de Embarque': ['Total General'], 'Cantidad de Pasajeros': [tabla_3['Cantidad de Pasajeros'].sum()]})
tabla_3 = pd.concat([tabla_3, fila_total], ignore_index=True)

display(tabla_3)
print("Tratamiento de faltantes: Los 2 registros con valor nulo en 'Embarked' se agruparon bajo la etiqueta 'Sin registrar' para mantener la consistencia con el total de 891 pasajeros.")

In [ ]:
tabla_4 = df.groupby('Pclass')['Fare'].agg(
    Cantidad_Pasajeros='count',
    Tarifa_Promedio='mean',
    Tarifa_Minima='min',
    Tarifa_Maxima='max'
).round(2).reset_index()

tabla_4['Pclass'] = tabla_4['Pclass'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'})
tabla_4.columns = ['Clase del Boleto', 'Pasajeros con Tarifa', 'Tarifa Promedio', 'Tarifa Mínima', 'Tarifa Máxima']

display(tabla_4)
print("Tratamiento de faltantes: No se detectaron valores faltantes en 'Fare' ni en 'Pclass', por lo que los cálculos incluyen a los 891 pasajeros.")

In [ ]:
tabla_5 = pd.crosstab(
    index=df['Pclass'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'}),
    columns=df['Embarked'].fillna('Sin registrar').map({'C': 'Cherburgo (C)', 'Q': 'Queenstown (Q)', 'S': 'Southampton (S)', 'Sin registrar': 'Sin registrar'}),
    margins=True,
    margins_name='Total'
)
tabla_5.index.name = 'Clase del Boleto'
tabla_5.columns.name = 'Puerto de Embarque'

display(tabla_5)
print("Tratamiento de faltantes: Los 2 pasajeros sin puerto registrado en 'Embarked' se incluyeron en la columna 'Sin registrar' para que la suma marginal coincida con los 891 pasajeros.")

In [ ]:
tabla_6 = pd.crosstab(
    index=df['Pclass'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'}),
    columns=df['Survived'].map({0: 'Falleció (0)', 1: 'Sobrevivió (1)'}),
    margins=True,
    margins_name='Total'
)
tabla_6.index.name = 'Clase del Boleto'
tabla_6.columns.name = 'Estado de Supervivencia'

display(tabla_6)
print("Tratamiento de faltantes: Ni 'Pclass' ni 'Survived' tienen valores nulos; el cruce abarca la totalidad de los 891 pasajeros.")

In [ ]:
total_familiares = df['SibSp'] + df['Parch']
categoria_familiar = pd.cut(
    total_familiares,
    bins=[-1, 0, 1, float('inf')],
    labels=['0 familiares (Viaja solo)', '1 familiar', '2 o más familiares']
)

tabla_7 = categoria_familiar.value_counts().sort_index().reset_index()
tabla_7.columns = ['Grupo Familiar a Bordo', 'Cantidad de Pasajeros']

fila_total = pd.DataFrame({'Grupo Familiar a Bordo': ['Total General'], 'Cantidad de Pasajeros': [tabla_7['Cantidad de Pasajeros'].sum()]})
tabla_7 = pd.concat([tabla_7, fila_total], ignore_index=True)

display(tabla_7)
print("Tratamiento de faltantes: Las columnas 'SibSp' y 'Parch' están completas (0 nulos), por lo que la suma de familiares se calculó para los 891 pasajeros.")

In [ ]:
def categorizar_edad(edad):
    if pd.isna(edad):
        return 'Sin registrar'
    elif edad < 18:
        return 'Menor de 18 años'
    else:
        return '18 años o más'

grupo_edad = df['Age'].apply(categorizar_edad)

tabla_8 = pd.crosstab(
    index=df['Pclass'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'}),
    columns=grupo_edad,
    margins=True,
    margins_name='Total'
)[['Menor de 18 años', '18 años o más', 'Sin registrar', 'Total']]

tabla_8.index.name = 'Clase del Boleto'
tabla_8.columns.name = 'Rango de Edad'

display(tabla_8)
print("Tratamiento de faltantes: Se identificaron 177 pasajeros con valor nulo en 'Age'. En lugar de eliminarlos o imputarlos, se clasificaron explícitamente en la categoría 'Sin registrar'.")

In [ ]:
conteo_tickets = df['Ticket'].value_counts()

tickets_distintos = len(conteo_tickets)
tickets_mas_de_uno = (conteo_tickets > 1).sum()
boleto_frecuente = conteo_tickets.idxmax()
pasajeros_boleto_frecuente = conteo_tickets.max()

boletos_empatados = ', '.join(conteo_tickets[conteo_tickets == pasajeros_boleto_frecuente].index.tolist())

tabla_9 = pd.DataFrame({
    'Indicador de Boletos': [
        'Códigos de boleto distintos',
        'Códigos asociados a más de un pasajero',
        'Código(s) que aparece(n) en más registros',
        'Cantidad de pasajeros en el boleto más frecuente'
    ],
    'Valor': [
        tickets_distintos,
        tickets_mas_de_uno,
        boletos_empatados,
        pasajeros_boleto_frecuente
    ]
})

display(tabla_9)
print("Tratamiento de faltantes: La columna 'Ticket' no presenta valores nulos (891 registros completos).")

In [ ]:
estado_cabina = df['Cabin'].notna().map({True: 'Con cabina registrada', False: 'Sin cabina registrada'})

tabla_10 = pd.crosstab(
    index=df['Pclass'].map({1: '1ª Clase', 2: '2ª Clase', 3: '3ª Clase'}),
    columns=estado_cabina,
    margins=True,
    margins_name='Total'
)[['Con cabina registrada', 'Sin cabina registrada', 'Total']]

tabla_10.index.name = 'Clase del Boleto'
tabla_10.columns.name = 'Estado de Cabina'

display(tabla_10)
print("Clase con más registros sin cabina: 3ª Clase (479 registros sin cabina).")
print("Tratamiento de faltantes: Los 687 valores nulos (NaN) en 'Cabin' no se descartaron; se contabilizaron explícitamente bajo la columna 'Sin cabina registrada'.")